In [ ]:
# imports and settings
import logging
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import gaussian_kde

import pcntoolkit.util.output
from pcntoolkit import (
    HBR,
    NormativeModel,
    NormData,
    SHASHbLikelihood,
    make_prior,
    plot_centiles,
    plot_qq,
)

sns.set_style("darkgrid")

# Suppress some annoying warnings and logs
pymc_logger = logging.getLogger("pymc")
pymc_logger.setLevel(logging.WARNING)
pymc_logger.propagate = False

warnings.simplefilter(action="ignore", category=FutureWarning)
pd.options.mode.chained_assignment = None
pcntoolkit.util.output.Output.set_show_messages(False)

In [ ]:
# Datasets 
meta_dir = Path("/home/traaffneu/bersal/digitalrodent/data/MetaData")
dr_dir = Path("/home/traaffneu/bersal/digitalrodent/results/dual_regression")

# name -> (metadata file, DR file). Check the names in the folders!
datasets = {
    "Awake":         ("Awake.tsv", "Awake.csv"),
    #"Awake_Rest":    ("Awake_Rest.tsv", "Awake_Rest.csv"),
    #"Psilocybin":    ("Psilocybin.tsv", "Psilocybin.csv"),
    "Douglas":       ("Douglas.tsv", "Douglas.csv"),
    "MPTP":          ("MPTP.tsv", "MPTP.csv"),
    "Sexdifference": ("Sexdifference.tsv", "Sexdifference.csv"),
}

exclude = []  # e.g. ["Awake"] to run without it
datasets = {k: v for k, v in datasets.items() if k not in exclude}

# Load metadata
meta_parts = []
for name, (meta_file, _) in datasets.items():
    part = pd.read_csv(meta_dir / meta_file, sep="\t")
    part["dataset"] = name
    meta_parts.append(part[["dataset", "Participant_Id", "Sex"]])

meta_df = pd.concat(meta_parts, ignore_index=True)

# Same formats for every dataset: ID as integer, sex as M/F
meta_df["Participant_Id"] = (
    meta_df["Participant_Id"].astype(str)
    .str.replace("sub-", "", regex=False)
    .str.split("_").str[0]
    .str.replace(r"(?<!\d)0+(?=\d)", "", regex=True)
)
meta_df["Sex"] = meta_df["Sex"].astype(str).str.strip().str.upper().str[0]

# One sex value per mouse
sex_df = meta_df.drop_duplicates()
assert not sex_df.duplicated(["dataset", "Participant_Id"]).any(), "some mouse has conflicting sex values"

print(sex_df.groupby("dataset")["Sex"].value_counts(dropna=False))

#Load DR components
dr_parts = []
for name, (_, dr_file) in datasets.items():
    # Comma-separated, with spaces before the commas
    part = pd.read_csv(dr_dir / dr_file, sep=r"\s*,\s*", engine="python")
    part["dataset"] = name
    dr_parts.append(part)

dr_df = pd.concat(dr_parts, ignore_index=True)

comp_cols = [c for c in dr_df.columns if c.startswith("comp")]
dr_df[comp_cols] = dr_df[comp_cols].apply(pd.to_numeric)

#ids = dr_df["subject"].str.extract(r"sub-(\d+)")[0]

#print("Scans without a sub-XXXX ID per dataset:")
#print(dr_df[ids.isna()].groupby("dataset").size())

# Example subject names from each dataset
#for name, group in dr_df.groupby("dataset"):
    #print(name, group["subject"].head(3).tolist())

# sub-0100100_ses-1_... -> 100100, to match the metadata
dr_df["Participant_Id"] = (
    dr_df["subject"].astype(str)
    .str.replace("sub-", "", regex=False)
    .str.split("_").str[0]
    .str.replace(r"(?<!\d)0+(?=\d)", "", regex=True)
)

# Sexdifference: S041 is labelled C10 in the DR file instead of C11 (typo)
dr_df.loc[(dr_df["dataset"] == "Sexdifference") & (dr_df["Participant_Id"] == "C10S41"), "Participant_Id"] = "C11S41"

print(dr_df.groupby("dataset").size())
display(dr_df.head())

In [ ]:
# Merge and checks
# Merge on dataset + ID, so equal IDs from different sites never mix
df = dr_df.merge(sex_df, on=["dataset", "Participant_Id"], how="left")
df["sex_encoded"] = df["Sex"].map({"M": 0, "F": 1}).astype("float64")
df["mouse_id"] = df["dataset"] + "_" + df["Participant_Id"].astype(str)

print(df["Sex"].value_counts(dropna=False))
print(len(df), "scans,", df["mouse_id"].nunique(), "mice")

# Mice not found in the metadata
unmatched = df.loc[df["Sex"].isna(), ["dataset", "mouse_id"]].drop_duplicates()
print("Unmatched mice per dataset:")
print(unmatched.groupby("dataset").size())

# --- temporary checks ---

print(meta_df.loc[meta_df["Participant_Id"].str.contains("c11", case=False), ["dataset", "Participant_Id"]])
print(meta_df.loc[meta_df["Participant_Id"].str.contains("S41"), ["dataset", "Participant_Id"]])

print(df.loc[df["Sex"].isna(), ["dataset", "Participant_Id"]].drop_duplicates())

# Awake_Rest: jgrAesAWc11L has no match in the metadata, dropped by the dropna below
print((dr_df["Participant_Id"] == "jgrAesAWc11L").sum(), "scans of jgrAesAWc11L")

df = df.dropna(subset=["Sex"] + comp_cols)

# Mice per dataset and sex
mice = df[["mouse_id", "dataset", "Sex"]].drop_duplicates()
print(pd.crosstab(mice["dataset"], mice["Sex"]))

# Scans per mouse
print(df.groupby("mouse_id").size().describe())

In [ ]:
# Ridge plot of all components by sex
sex_colors = {"M": "tab:blue", "F": "tab:orange"}
fig, ax = plt.subplots(figsize=(9, 12))

x_grid = np.linspace(df[comp_cols].min().min(), df[comp_cols].max().max(), 500)
rng = np.random.default_rng(0)

for i, comp in enumerate(comp_cols[::-1]):  # comp1 at the top
    ax.axhline(i, color="grey", lw=1)

    for j, (sex, color) in enumerate(sex_colors.items()):
        values = df.loc[df["Sex"] == sex, comp].values
        kde = gaussian_kde(values)(x_grid)
        kde = kde / kde.max() * 0.8

        ax.fill_between(x_grid, i, i + kde, color=color, alpha=0.4)
        ax.plot(x_grid, i + kde, color=color, lw=1)

        # Points: M slightly lower, F slightly higher, so they don't overlap
        jitter = rng.uniform(0.02, 0.1, size=len(values)) + j * 0.1
        ax.scatter(values, i + jitter, s=6, color=color, alpha=0.5, zorder=3)

        # Median per sex
        med = np.median(values)
        ax.plot([med, med], [i, i + 0.6], color=color, lw=2)

ax.set_yticks(range(len(comp_cols)))
ax.set_yticklabels(comp_cols[::-1])
ax.set_xlabel("DR value")
ax.set_ylabel("Component")
ax.set_title("All datasets: dual regression component means by sex")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.5) for c in sex_colors.values()],
          labels=list(sex_colors.keys()))
plt.tight_layout()
plt.show()

In [ ]:
# Split train/test by mouse, stratified by dataset and sex
rng = np.random.default_rng(42)  # fixed seed
test_frac = 0.2

test_mice = []
for _, group in mice.groupby(["dataset", "Sex"]):
    ids = group["mouse_id"].values
    n_test = int(round(test_frac * len(ids)))
    test_mice.extend(rng.choice(ids, size=n_test, replace=False))

is_test = df["mouse_id"].isin(test_mice)
train_df = df[~is_test].copy()
test_df = df[is_test].copy()

print("Train:", len(train_df), "scans,", train_df["mouse_id"].nunique(), "mice")
print("Test: ", len(test_df), "scans,", test_df["mouse_id"].nunique(), "mice")
print("Overlap:", set(train_df["mouse_id"]) & set(test_df["mouse_id"]))


# NormData
covariates = ["sex_encoded"]
batch_effects = ["dataset"]
feature_to_model = comp_cols[:2]  # test with 2 components first, then comp_cols

train = NormData.from_dataframe(
    name="train",
    dataframe=train_df,
    covariates=covariates,
    batch_effects=batch_effects,
    response_vars=feature_to_model,
    remove_Nan=False,
)
test = NormData.from_dataframe(
    name="test",
    dataframe=test_df,
    covariates=covariates,
    batch_effects=batch_effects,
    response_vars=feature_to_model,
    remove_Nan=False,
)

# Visualize one component as a check
feature_to_plot = feature_to_model[0]
fig, ax = plt.subplots(1, 2, figsize=(15, 5))

sns.stripplot(data=train_df, x="dataset", y=feature_to_plot, hue="Sex", dodge=True, alpha=0.4, ax=ax[0])
ax[0].set_title(f"{feature_to_plot} by dataset and sex (train)")

sns.histplot(data=train_df, x=feature_to_plot, hue="dataset", kde=True, ax=ax[1])
ax[1].set_title(f"{feature_to_plot} distribution per dataset (train)")

plt.tight_layout()
plt.show()

In [ ]:
# Initialize and Fit Normative Model: HBR with SHASH likelihood
# Mean: linear in sex, intercept varies per dataset
mu = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 10.0)),
    intercept=make_prior(
        random=True,
        mu=make_prior(dist_name="Normal", dist_params=(0.0, 1.0)),
        sigma=make_prior(dist_name="Normal", dist_params=(0.0, 1.0), mapping="softplus", mapping_params=(0.0, 3.0)),
    ),
)

# Spread: linear in sex, intercept varies per dataset
sigma = make_prior(
    linear=True,
    slope=make_prior(dist_name="Normal", dist_params=(0.0, 2.0)),
    intercept=make_prior(
        random=True,
        mu=make_prior(dist_name="Normal", dist_params=(1.0, 1.0)),
        sigma=make_prior(dist_name="Normal", dist_params=(0.0, 1.0), mapping="softplus", mapping_params=(0.0, 3.0)),
    ),
    mapping="softplus",
    mapping_params=(0.0, 3.0),
)

# Skewness and tail heaviness
epsilon = make_prior(dist_name="Normal", dist_params=(0.0, 1.0))
delta = make_prior(dist_name="Normal", dist_params=(1.0, 1.0), mapping="softplus", mapping_params=(0.0, 3.0, 0.6))

model = NormativeModel(
    HBR(
        name="hbr_shashb",
        likelihood=SHASHbLikelihood(mu, sigma, epsilon, delta),
        draws=500,  # 1500 for the final run
        tune=500,
        chains=4,
        cores=4,
        progressbar=True,
    ),
    inscaler="standardize",
    outscaler="standardize",
)

model.fit_predict(train, test)

In [ ]:

# Evaluation
#in NormData I selected only comp1 and comp 2 for now
# Centile plots
for comp in feature_to_model:
    plot_centiles(
        model,
        scatter_data=train,
        response_vars=[comp],
        centiles=[0.05, 0.5, 0.95],
        covariate="sex_encoded",
        scatter_kwargs={"s": 30, "alpha": 0.3},
    )

In [ ]:
# Statistics
print("Train set statistics:")
display(train.get_statistics_df().loc[feature_to_model])
print("Test set statistics:")
display(test.get_statistics_df().loc[feature_to_model])

In [ ]:
# QQ plots
for comp in feature_to_model:
    plot_qq(test.sel({"response_vars": [comp]}), plot_id_line=True)